In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import psi4
import fortecubeview
import shutil
from scipy.optimize import curve_fit
from pib_helper import create_psi4_molecule

<a id="part3"></a>

## Part 3 — Computational Chemistry

So far we have worked with the particle-in-a-box as a simplified model. Now we use quantum chemistry software to compute the electronic structure of **real molecules** and see how the model holds up.

We use the **Hartree–Fock (Self-Consistent Field, SCF)** method, which handles many interacting electrons by assuming each electron moves in the average field created by all the others. The procedure is:

1. **Initial guess** — build approximate orbitals from a set of basis functions (the **basis set**).
2. **Build the Fock operator** — an effective one-electron Hamiltonian including kinetic energy, nuclear attraction, and mean-field electron repulsion:
$$\hat{F}\psi_i = \varepsilon_i \psi_i$$
3. **Solve for new orbitals** — find orbital shapes and energies.
4. **Iterate** — repeat until the orbitals stop changing (**convergence**).

This is the same strategy as Part 1: start from a guess, minimize the energy, and let the physics determine the answer.

### HOMO–LUMO Gap

Each orbital holds two electrons (spin-up and spin-down). The highest filled orbital is the **HOMO** and the lowest empty orbital is the **LUMO**. Their energy difference

$$\Delta E = E_{\text{LUMO}} - E_{\text{HOMO}}$$

determines the energy of light the molecule can absorb. In Part 4 we will compare these gaps to the particle-in-a-box prediction.

---

### Coding Activity 3
`30 points`

- C4.3: Apply the particle-in-a-box model to conjugated polyenes.
- P4.3: Perform quantum chemical calculations with `psi4`.

#### Part A — Running a Hartree–Fock calculation

The cell below runs an HF/STO-3G calculation on **ethane** and plots its orbital energies. Run it and **add a comment to every line** explaining what it does.

Then, in the next cell, run the same calculation on **octatetraene** (`C=CC=CC=CC=C`) and plot its orbital energies.

In [ ]:
# --- Worked example: ethane ---
psi4.set_output_file('output.dat', False)
psi4.set_memory('1 GB')

mol_ethane = create_psi4_molecule('CC')
energy_ethane, wfn_ethane = psi4.energy('SCF/STO-3G', return_wfn=True, molecule=mol_ethane)
print(f'Ethane HF/STO-3G energy: {energy_ethane:.6f} a.u.')

# Plot all orbital energies
eps_ethane = wfn_ethane.epsilon_a().np
plt.plot(eps_ethane, 'o', markersize=5, color='steelblue')
plt.axhline(0, color='grey', linewidth=0.5)
plt.xlabel('Orbital index')
plt.ylabel('Energy (a.u.)')
plt.title('Ethane: All HF/STO-3G Orbital Energies')
plt.show()

In [ ]:
# YOUR CODE HERE
# Run HF/STO-3G on octatetraene (C=CC=CC=CC=C)
# Store the wavefunction in wfn_oct — you will need it in Parts B and C
# Plot all orbital energies


#### Part B — Visualizing molecular orbitals

The cell below generates cube files for orbitals near the HOMO and LUMO of octatetraene and displays them with `fortecubeview`. Run it, then examine the orbital shapes.

Octatetraene (C$_8$H$_{10}$) has $N_e = 58$ electrons, so the HOMO is orbital index 28 (zero-indexed).

**Your task:** identify which of the displayed orbitals are **$\pi$ orbitals** — they have lobes above and below the molecular plane. Record their indices; you will use them in Part C.

In [ ]:
# --- Given: orbital visualization ---
import os
os.makedirs('cubes', exist_ok=True)

# psi4 CUBEPROP_ORBITALS uses 1-indexed orbital numbers
homo_psi4 = 29  # octatetraene HOMO (1-indexed; 58 electrons / 2 = 29)
orb_numbers = list(range(homo_psi4 - 3, homo_psi4 + 4))  # [26, 27, 28, 29, 30, 31, 32]

psi4.set_options({
    'CUBEPROP_TASKS': ['orbitals'],
    'CUBEPROP_FILEPATH': 'cubes',
    'CUBEPROP_ORBITALS': orb_numbers,
})
psi4.cubeprop(wfn_oct)
fortecubeview.plot('cubes', colorscheme='wow')

In [ ]:
# Run this cell when you are done inspecting orbitals
shutil.rmtree('cubes', ignore_errors=True)

In [ ]:
# Record the orbital indices that you identified as pi orbitals:
pi_indices = []  # YOUR CODE HERE — e.g. [25, 26, 27, 28, 29, 30, 31]

#### Part C — Fitting the PIB model to $\pi$ orbital energies

Plot only the $\pi$ orbital energies, then try to fit the particle-in-a-box model $E = A \cdot n^2$ to them using `curve_fit`. Does the PIB model capture the pattern?

In [ ]:
# YOUR CODE HERE
# 1. eps = wfn_oct.epsilon_a().np
# 2. pi_energies = eps[pi_indices] * 27.211  (convert to eV)
# 3. n_pi = np.arange(1, len(pi_energies) + 1)
# 4. Plot pi_energies vs n_pi
# 5. Define: def pib_model(n, A): return A * n**2
# 6. Fit with curve_fit and plot the fit on top of the data

### Question 3
`15 points`

- C4.2: Explain features of particle-in-a-box wavefunctions and their energies.
- C4.4: Evaluate the tradeoffs of different physical models.

a) Compare the Hartree–Fock procedure to the variational minimization you did in Part 1. What is conceptually the same? What is different?

b) Look at the $\pi$ orbitals of octatetraene. In what ways do they resemble the particle-in-a-box wavefunctions from Part 2?

c) How well does the PIB model $E = An^2$ fit the $\pi$ orbital energies? What does this tell you about the limitations of the particle-in-a-box as a model for real molecules?

---

*Your answer here (`double click me!`):*

---